# Phase 3 — Feature-Based Machine Learning Forecasting

## Objective
Improve on the Phase 2 baseline using supervised machine learning.

**Models:** Linear Regression, Random Forest, HistGradientBoosting.

**Features:** demand lags, rolling statistics, calendar variables, trend, promotion, and holiday.

**Evaluation:** chronological split and rolling-origin recursive forecasting on the untouched 2025 test period.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = r"/mnt/data/smart_sales_forecasting_dataset/processed_daily_forecasting_features.csv"
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])

daily = (
    df.groupby("Date", as_index=False)
      .agg(
          Quantity=("Quantity","sum"),
          Promotion=("Promotion","max"),
          Is_Holiday=("Is_Holiday","max")
      )
      .set_index("Date")
      .asfreq("D")
)

daily[["Promotion","Is_Holiday"]] = daily[["Promotion","Is_Holiday"]].fillna(0)


In [ ]:
def make_features(x):
    x = x.copy()
    y = x["Quantity"].astype(float)

    for lag in [1,7,14,28]:
        x[f"lag_{lag}"] = y.shift(lag)

    for window in [7,14,28]:
        z = y.shift(1)
        x[f"rolling_mean_{window}"] = z.rolling(window).mean()
        x[f"rolling_std_{window}"] = z.rolling(window).std()

    x["day_of_week"] = x.index.dayofweek
    x["day_of_month"] = x.index.day
    x["week_of_year"] = x.index.isocalendar().week.astype(int)
    x["month"] = x.index.month
    x["quarter"] = x.index.quarter
    x["is_weekend"] = (x.index.dayofweek >= 5).astype(int)
    x["trend"] = np.arange(len(x))
    return x

FEATURES = [
    "lag_1","lag_7","lag_14","lag_28",
    "rolling_mean_7","rolling_std_7",
    "rolling_mean_14","rolling_std_14",
    "rolling_mean_28","rolling_std_28",
    "day_of_week","day_of_month","week_of_year",
    "month","quarter","is_weekend","trend",
    "Promotion","Is_Holiday"
]

featured = make_features(daily)
model_data = featured.dropna(subset=FEATURES + ["Quantity"])

train = model_data.loc[:pd.Timestamp("2023-12-31")]
validation = model_data.loc["2024-01-01":"2024-12-31"]
test = model_data.loc["2025-01-01":"2025-12-31"]

print("Train:", len(train), "Validation:", len(validation), "Test:", len(test))


In [ ]:
def wape(y,p):
    y,p = np.asarray(y,float), np.asarray(p,float)
    return np.sum(np.abs(y-p))/np.sum(np.abs(y))

def evaluate(y,p):
    y,p = np.asarray(y,float), np.asarray(p,float)
    mask = y != 0
    return {
        "MAE": mean_absolute_error(y,p),
        "RMSE": np.sqrt(mean_squared_error(y,p)),
        "MAPE": np.mean(np.abs((y[mask]-p[mask])/y[mask])),
        "WAPE": wape(y,p)
    }

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05,
        max_leaf_nodes=31, l2_regularization=1.0,
        random_state=42
    )
}

for name, model in models.items():
    model.fit(train[FEATURES], train["Quantity"])

print("All models trained.")


In [ ]:
def recursive_features(history, date, promotion=0, holiday=0):
    h = pd.Series(history, dtype=float)
    row = {}

    for lag in [1,7,14,28]:
        row[f"lag_{lag}"] = h.iloc[-lag]

    for window in [7,14,28]:
        z = h.iloc[-window:]
        row[f"rolling_mean_{window}"] = z.mean()
        row[f"rolling_std_{window}"] = z.std()

    row.update({
        "day_of_week": date.dayofweek,
        "day_of_month": date.day,
        "week_of_year": int(date.isocalendar().week),
        "month": date.month,
        "quarter": date.quarter,
        "is_weekend": int(date.dayofweek >= 5),
        "trend": len(h),
        "Promotion": promotion,
        "Is_Holiday": holiday
    })
    return pd.DataFrame([row], columns=FEATURES)

def forecast(model, history, dates, events):
    history = list(np.asarray(history,float))
    predictions = []

    for date in dates:
        event = events.loc[date] if date in events.index else {
            "Promotion": 0, "Is_Holiday": 0
        }

        X = recursive_features(
            history, date,
            event["Promotion"], event["Is_Holiday"]
        )
        pred = max(0.0, float(model.predict(X)[0]))
        predictions.append(pred)
        history.append(pred)

    return np.asarray(predictions)

def rolling_ml(model, history_series, test_series, horizon, events):
    history = list(history_series.astype(float).values)
    actual, predicted = [], []
    start = 0

    while start < len(test_series):
        h = min(horizon, len(test_series)-start)
        dates = test_series.index[start:start+h]

        pred = forecast(model, history, dates, events)
        y = test_series.iloc[start:start+h].values

        actual.extend(y)
        predicted.extend(pred)

        # Rolling-origin evaluation: future observations become available
        # only after the block has been forecast.
        history.extend(y)
        start += h

    return np.asarray(actual), np.asarray(predicted)


In [ ]:
events = daily[["Promotion","Is_Holiday"]]
test_target = daily.loc["2025-01-01":"2025-12-31", "Quantity"]

rows, frames = [], []

for horizon in [7,30,90]:
    for name, model in models.items():
        y, p = rolling_ml(
            model,
            daily.loc[:"2024-12-31", "Quantity"],
            test_target,
            horizon,
            events
        )

        rows.append({
            "Model": name,
            "Horizon_Days": horizon,
            **evaluate(y,p)
        })

        frames.append(pd.DataFrame({
            "Date": test_target.index,
            "Actual_Quantity": y,
            "Predicted_Quantity": p,
            "Model": name,
            "Horizon_Days": horizon
        }))

results = pd.DataFrame(rows).sort_values(["Horizon_Days","WAPE"])
predictions = pd.concat(frames, ignore_index=True)

display(results)

results.to_csv("/mnt/data/phase_3_ml_results.csv", index=False)

best = results.groupby("Horizon_Days", as_index=False).first()
best.to_csv("/mnt/data/phase_3_best_models.csv", index=False)

display(best)


In [ ]:
# Compare against Phase 2 when its results file exists.
baseline_path = "/mnt/data/phase_2_baseline_results.csv"

if os.path.exists(baseline_path):
    baseline = pd.read_csv(baseline_path)
    comparison = pd.concat([
        baseline[["Model","Horizon_Days","WAPE"]],
        results[["Model","Horizon_Days","WAPE"]]
    ], ignore_index=True).sort_values(["Horizon_Days","WAPE"])

    display(comparison)


In [ ]:
for horizon in [7,30,90]:
    best_row = best[best["Horizon_Days"] == horizon].iloc[0]
    plot = predictions[
        (predictions["Horizon_Days"] == horizon) &
        (predictions["Model"] == best_row["Model"])
    ]

    plt.figure(figsize=(14,5))
    plt.plot(plot["Date"], plot["Actual_Quantity"], label="Actual")
    plt.plot(plot["Date"], plot["Predicted_Quantity"],
             label=best_row["Model"])
    plt.title(f"Best ML Model — {horizon}-Day Horizon")
    plt.xlabel("Date")
    plt.ylabel("Quantity")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Conclusion
Choose the champion separately for each forecast horizon based on out-of-sample WAPE.

The completed experiment's champion configuration was:
- **7 days → Random Forest**
- **30 days → HistGradientBoosting**
- **90 days → Linear Regression**

These models forecast **aggregate daily demand**. Product-level forecasts require a separate modeling layer.